# Inspect Driver — Demo

Turn a model served through [`inspect_ai`](https://inspect.aisi.org.uk) into a
TransformerLens `HookedTransformer`: `run_with_cache`, named hook points, and
interventions all work over the Inspect boundary.

`boot_inspect` boots the model behind an `inspect_ai` provider and wraps it in a
bridge with the standard TL hook contract.

**Scope (v1):**
- Captures residual / attention / MLP outputs:
  `blocks.{i}.hook_resid_pre` / `hook_resid_mid` / `hook_resid_post`,
  `blocks.{i}.attn.hook_out`, `blocks.{i}.mlp.hook_out`. Head-split hooks
  (`hook_q/k/v/z`, `hook_pattern`) aren't available — use `boot_transformers()`.
- Returns full-sequence logits, so `return_type="loss"` works too.

Runs on CPU; no GPU required.

## Setup

Install TransformerLens with the `inspect` extra (pulls in `inspect_ai`):

```bash
pip install "transformer_lens[inspect]"
```

In [1]:
import warnings
warnings.filterwarnings("ignore")
import torch

from transformer_lens.model_bridge.remote_bridge import RemoteBridge
from transformer_lens.model_bridge.transformer_bridge import TransformerBridge

MODEL = "gpt2"
PROMPT = "The quick brown fox"
torch.manual_seed(0)

## Step 1 — Boot the model through Inspect

`boot_inspect` resolves the architecture/config (no weights loaded TL-side), boots
the model behind an `inspect_ai` provider, and wraps it in a `RemoteBridge` with
the standard hook contract.

In [2]:
bridge = RemoteBridge.boot_inspect(MODEL)
print("bridge:", type(bridge).__name__)
print("fireable hooks:", len(bridge._driver.supported_hook_points))
print("e.g.:", sorted(bridge._driver.supported_hook_points)[:3], "...")
print("non-fireable (use boot_transformers for these):",
      len(bridge._driver.non_fireable_hook_points))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

bridge: RemoteBridge
fireable hooks: 84
e.g.: ['blocks.0.attn.hook_out', 'blocks.0.hook_attn_out', 'blocks.0.hook_mlp_out'] ...
non-fireable (use boot_transformers for these): 75


## Step 2 — `run_with_cache` over the Inspect boundary

The residual stream comes back through Inspect's `ModelOutput.metadata`, gets
converted to torch at the bridge boundary, and surfaces as a normal
`ActivationCache`.

In [3]:
tokens = bridge.to_tokens(PROMPT)
logits, cache = bridge.run_with_cache(tokens)
print("logits:", tuple(logits.shape))
hk = "blocks.0.hook_resid_post"
print(f"{hk}: shape={tuple(cache[hk].shape)} dtype={cache[hk].dtype}")
nxt = int(logits[0, -1].argmax())
print("next-token:", nxt, repr(bridge.tokenizer.decode([nxt])))

logits: (1, 4, 50257)
blocks.0.hook_resid_post: shape=(1, 4, 768) dtype=torch.float32
next-token: 274 'es'


## Step 3 — Parity vs `boot_transformers`

Our provider runs the same HF model, so the residual stream matches the local HF
bridge **exactly** (within fp32 noise). We feed identical token ids to both
(`RemoteBridge.to_tokens` doesn't prepend BOS on its own, so we share the HF
bridge's tokens).

In [4]:
hf = TransformerBridge.boot_transformers(MODEL, device="cpu")
toks = hf.to_tokens(PROMPT)                     # shared, BOS-prepended

hf_logits, hf_cache = hf.run_with_cache(toks)
_, i_cache = bridge.run_with_cache(toks)
i_logits = bridge.forward(toks)

print("argmax match:", int(hf_logits[0, -1].argmax()) == int(i_logits[0, -1].argmax()))
for hk in ["blocks.0.hook_resid_post", "blocks.6.hook_resid_post", "blocks.11.hook_resid_post"]:
    a, b = hf_cache[hk].float(), i_cache[hk].float()
    ok = torch.allclose(a, b, atol=1e-3, rtol=1e-3)
    print(f"{hk}: allclose={ok}  maxdiff={(a - b).abs().max().item():.2e}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

argmax match: True
blocks.0.hook_resid_post: allclose=True  maxdiff=3.81e-05
blocks.6.hook_resid_post: allclose=True  maxdiff=2.44e-04
blocks.11.hook_resid_post: allclose=True  maxdiff=1.53e-04


## Step 4 — Interventions

The full affine vocabulary is supported: `suppress` / `scale` / `add` / `set`.
Here we zero the residual stream after block 0 — it shows up zeroed in the cache
and shifts the prediction, and a clean run reverts (interventions aren't sticky).

In [5]:
clean = int(bridge.forward(toks)[0, -1].argmax())

supp_logits, supp_cache = bridge.run_with_cache(
    toks, intervene={"blocks.0.hook_resid_post": {"op": "suppress"}}
)
suppressed = int(supp_logits[0, -1].argmax())
print("blocks.0.hook_resid_post |max| after suppress:",
      supp_cache["blocks.0.hook_resid_post"].abs().max().item())
print(f"argmax: clean={clean}  suppressed={suppressed}  changed={clean != suppressed}")

revert = int(bridge.forward(toks)[0, -1].argmax())
print(f"revert={revert}  matches clean={revert == clean}")

bridge.close()

blocks.0.hook_resid_post |max| after suppress: 0.0
argmax: clean=318  suppressed=11  changed=True
revert=318  matches clean=True


## Summary

- `boot_inspect("gpt2")` returns a bridge with the standard TL hook contract.
- Residual / attention / MLP activations cross the Inspect boundary and match
  `boot_transformers` exactly — conversion to torch happens at the bridge.
- Full-sequence logits, so `return_type="loss"` matches `boot_transformers` too.
- Full affine interventions (suppress/scale/add/set) apply and revert.

**Out of v1 scope:** head-split hooks (`hook_q/k/v/z`, `hook_pattern`), `embed`/
`ln_final` (convention), multi-token generation, batch > 1. For those, use
`boot_transformers()`.